## Soru 2: Numpy `linalg.eig` Fonksiyonunun İncelenmesi ve Uygulanması

Bu bölümde Numpy kütüphanesinin doğrusal cebir modülünde yer alan `eig` fonksiyonunun çalışma mantığı, dokümantasyonu ve kaynak kodları inceledim.

### 1. Dokümantasyon İncelemesi (Girdi ve Çıktılar)
Numpy dokümantasyonuna göre `w, v = numpy.linalg.eig(a)` fonksiyonu şu şekilde çalışır:
* **Girdi (`a`):** Fonksiyon parametre olarak karesel (square) bir matris bekler. Vektörlerin aynı uzayda dönüştürülebilmesi için satır ve sütun sayılarının eşit olması zorunludur.
* **Çıktılar:** Fonksiyon geriye bir tuple (ikili) döndürür:
  * **`w` (Özdeğerler):** Matrisin özdeğerlerini içeren 1 boyutlu bir dizidir.
  * **`v` (Özvektörler):** Bu matrisin **sütunları**, `w` dizisindeki özdeğerlere karşılık gelen normalize edilmiş özvektörleri temsil eder. Yani `w[i]` özdeğerinin özvektörü `v[:, i]` sütunudur (satırı değil).

### 2. Kaynak Kod İncelemesi (Arka Plan İşlemleri)
Numpy'ın GitHub üzerindeki kaynak kodları incelendiğinde, özdeğer hesaplamalarının doğrudan saf Python döngüleri ile yapılmadığı görülmektedir. Python hesaplama açısından yavaş bir dil olduğundan, `eig` fonksiyonu bir arayüz (wrapper) görevi görür.
* Arka planda, C ve Fortran ile yazılmış, donanım seviyesinde inanılmaz derecede optimize edilmiş **LAPACK (Linear Algebra PACKage)** kütüphanesi çağrılır.
* Spesifik olarak matrisin türüne göre LAPACK'in `geev` gibi özel rutinleri kullanılarak işlemler çok yüksek hızda gerçekleştirilir ve sonuçlar Python'a geri döndürülür.

In [ ]:
import numpy as np

# 1. Karesel bir örnek matris oluşturalım
A = np.array([[4, -2],
              [1,  1]])

print("Orijinal Matris (A):")
print(A)
print("\n")

# 2. linalg.eig fonksiyonunu kullanarak özdeğer ve özvektörleri hesaplayalım
ozdegerler, ozvektorler = np.linalg.eig(A)

# 3. Sonuçları ekrana yazdıralım
print("Özdeğerler (Eigenvalues):")
print(ozdegerler)
print("\n")

print("Özvektörler Matrisi (Eigenvectors):")
print(ozvektorler)
print("\n")

# 4. Dokümantasyonda belirtilen "sütun" kuralını gösterelim
print("Detaylı İnceleme")
for i in range(len(ozdegerler)):
    print(f"{i+1}. Özdeğer: {ozdegerler[i]}")
    # Özvektörleri satır olarak değil, sütun ([:, i]) olarak çekiyoruz
    print(f"{i+1}. Özdeğere ait Özvektör: {ozvektorler[:, i]}\n")

Orijinal Matris (A):
[[ 4 -2]
 [ 1  1]]


Özdeğerler (Eigenvalues):
[3. 2.]


Özvektörler Matrisi (Eigenvectors):
[[0.89442719 0.70710678]
 [0.4472136  0.70710678]]


Detaylı İnceleme
1. Özdeğer: 3.0
1. Özdeğere ait Özvektör: [0.89442719 0.4472136 ]

2. Özdeğer: 2.0
2. Özdeğere ait Özvektör: [0.70710678 0.70710678]



In [5]:
import numpy as np

#KENDİ ÖZDEĞER HESAPLAMA FONKSİYONUMUZ (QR İterasyonu)
def kendi_ozdeger_fonksiyonumuz(A, iterasyon_sayisi=100):
    # Orijinal matrisi bozmamak için kopyasını alıyoruz
    Ak = np.copy(A)
    
    # Belirlediğimiz sayı kadar iterasyon (döngü) yapıyoruz
    for _ in range(iterasyon_sayisi):
        # Matrisi Q ve R olarak ikiye ayır
        Q, R = np.linalg.qr(Ak)
        # Ters sırayla çarparak matrisi güncelle (R * Q)
        Ak = np.dot(R, Q)
        
    # Yeterli iterasyon sonrası ana köşegendeki (diagonal) elemanlar özdeğerlerimizdir
    ozdegerler = np.diag(Ak)
    return ozdegerler


#TEST VE KARŞILAŞTIRMA AŞAMASI

# Karesel bir örnek matris oluşturalım
A = np.array([[4.0, 1.0],
              [2.0, 3.0]])

print("Orijinal Matris (A):")
print(A)
print("\n")

# Kendi yazdığımız fonksiyon ile hesaplama
bizim_sonuclar = kendi_ozdeger_fonksiyonumuz(A, iterasyon_sayisi=50)
print(f"BİZİM ALGORİTMAMIZIN BULDUĞU ÖZDEĞERLER: \n{bizim_sonuclar}")
print("\n")

# Numpy'ın hazır fonksiyonu ile hesaplama
numpy_ozdegerler, numpy_ozvektorler = np.linalg.eig(A)
print(f"NUMPY'IN (np.linalg.eig) BULDUĞU ÖZDEĞERLER: \n{numpy_ozdegerler}")
print("\n")

# Karşılaştırma Sonucu
fark = np.abs(bizim_sonuclar - numpy_ozdegerler)
print(f"İki yöntem arasındaki fark: \n{fark}")

Orijinal Matris (A):
[[4. 1.]
 [2. 3.]]


BİZİM ALGORİTMAMIZIN BULDUĞU ÖZDEĞERLER: 
[5. 2.]


NUMPY'IN (np.linalg.eig) BULDUĞU ÖZDEĞERLER: 
[5. 2.]


İki yöntem arasındaki fark: 
[3.55271368e-15 0.00000000e+00]
